In [2]:
%load_ext autoreload
%autoreload 2 

# AISDI - ćwiczenie 2
## Wprowadzenie
Celem drugiego zadania z przedmiotu Algorytmy i Struktury Danych (AISDI) była implementacja oraz analiza wydajności algorytmów grafowych na przykładzie miejskiej sieci drogowej. Głównym zadaniem było zaimportowanie danych zawierających odległości oraz czasy przejazdu między węzłami w różnych porach dnia, a następnie wyznaczenie optymalnych tras na podstawie kryterium dystansu oraz czasu.

## Zaimportowane biblioteki
Zgodnie z wymaganiami projektowymi, do reprezentacji grafu nie wykorzystano gotowych bibliotek (takich jak networkx). Graf został zaimplementowany od podstaw przy użyciu zmodyfikowanej listy sąsiedztwa.

Główną strukturą przechowującą dane jest słownik, którego kluczem jest identyfikator węzła (miasta), a wartością lista obiektów klasy Destination.
Zastosowane narzędzia i moduły:

- csv – do wczytywania i parsowania plików z danymi.

- destination.py – autorska klasa reprezentująca krawędź grafu (cel podróży) wraz z jej wagami (dystans, czas w zależności od pory dnia).

- graph_logic.py – moduł zawierający implementację algorytmów trasowania.

- time – do przeprowadzania pomiarów wydajności i czasu kompilacji algorytmów.

In [3]:
import csv
from pprint import pprint
from destination import Destination
from graph_logic import *
import time

## Analiza danych wejściowych
Dane wejściowe pochodzą z pliku city_small.csv. Każdy wiersz reprezentuje krawędź grafu nieskierowanego (dwukierunkową drogę) wraz z zestawem wag. Struktura danych prezentuje się następująco:

In [4]:
with open("data/city_small.csv", newline="") as csv_file:
    small_cities = csv.reader(csv_file)
    for index, row in enumerate(small_cities):
        if index > 5:
            break
        print(row)

['od', 'do', 'odleglosc', 'czas_rano', 'czas_poludnie', 'czas_wieczor']
['0', '9', '17.1', '19', '11', '14']
['0', '20', '6.84', '11', '5', '8']
['1', '11', '9.4', '22', '12', '12']
['1', '21', '8.12', '13', '11', '15']
['1', '22', '19.55', '33', '20', '20']


Konstrukcja grafu odbywa się poprzez iterację po pliku CSV i dwukrotne dodanie krawędzi (od A do B oraz od B do A) do słownika pathways za pomocą metody create_destination().


## Wyznaczanie najkrótszej ścieżki
W pierwszej fazie badań przetestowano działanie klasycznego algorytmu Dijkstry, szukającego najkrótszej ścieżki od wybranego wierzchołka do pozostałych węzłów na podstawie wag odległościowych. Zbadano zachowanie algorytmu w zależności od rozmiaru grafu.

### Małe miasto
Algorytm dla małej siatki skrzyżowań odnalazł trasy błyskawicznie. W wynikach prawidłowo zidentyfikowano braki połączeń między odizolowanymi węzłami (oznaczone wartością inf), co świadczy o tym, że badany podgraf nie jest w pełni spójny.

In [ ]:

small_pathways = import_city("data/city_small.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(49, small_pathways)
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra is {total_time:.04f}")


Distance from 49 to 37 is 60.63
Distance from 17 to 32 is 136.76
Distance from 40 to 34 is 26.46
Distance from 47 to 31 is 101.63
Distance from 21 to 9 is inf
Distance from 17 to 40 is 75.74
Distance from 20 to 37 is inf
Distance from 27 to 29 is 82.45
Distance from 16 to 28 is inf
Distance from 40 to 43 is 51.79

Total time for small city dijkstra is 0.0005


Na tym etapie porównamy średni czas podróży 
Algorytm wykonuje się stosunkowo szybko.
### Czas
- zaprezentowano algortym osobnej dijstry dla czasu
- liczy od w trakcie godzine/czas obecny dla grafu
- w trakcie iteracji po grafie zmienia godzine i od niej pobiera odpowiedni czas

In [11]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(49, small_pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 48 to 17 is 10:55
Arrival time from 42 to 33 is 08:03
Arrival time from 46 to 19 is 07:52
Arrival time from 20 to 5 is inf
Arrival time from 14 to 39 is 09:46
Arrival time from 5 to 5 is 07:30
Arrival time from 27 to 14 is 08:05
Arrival time from 11 to 17 is 09:14
Arrival time from 8 to 47 is 08:11
Arrival time from 28 to 10 is 10:50

Total time for small city dijkstra time is 0.0007
This simulation is for starting time equal 07:30


### Komiwojazer:
Ten etap był najtrudniejszy do zrealizowania, po pierwsze w wielu przypadkach dotyczących grafu nie mam możliwości przejscia z jednegoe miasta do drugiego bezposrednio, dlatego trzeba zawracać. Dodatkowo niektóre miasta  tworzą graf, który jest odizolowany od głownego grafu, co powoduje, że klaysyczne rozwiązanie problemu kowimojażera staje się niemożliwe, a do tego należy wspomnieć, że tym przypadku została użyta heurystyka, bo dla zwykłego komputeraz obliczenie poprawnej najkrótszej drogi zajmuje n! czasu, dla kilku miast komputer szybko znajdzie najlepszą trasę, ale dla kilkudziesięciu miast potrzebny czas wynosi więcej niż czas istnienie wszechświata. W tym przypadku została użyta najprostrza heurystyka, która nie jest optymalna.

Tak prezentują się wyniki heurystyki, w której to algortym szuka najkrótszej drogi dla grafu małego miasta:

In [12]:
start_time = time.perf_counter()
start_city = "21"
road, distance = distance_of_travelling_salesman(small_pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

Cities that were visited from 21 are: 
['21', '39', '1', '34', '25', '42', '11', '4', '33', '44', '13', '6', '22', '23', '47', '29', '5', '8', '45', '28', '17', '40', '24', '37', '46', '38', '43', '48', '30', '19', '14', '18', '10', '15', '32', '2', '27', '7', '31', '49', '12', '36', '21']
and the distance is 631.46
It took programme 0.03213429206516594 for solving travelling salesman problem


## Średnie Miasto